# NLPGym – Guia Completo
Este notebook traz uma introdução conceitual e exemplos práticos **robustos** de uso do **NLPGym** para aplicar Aprendizado por Reforço (RL) em tarefas de NLP.

## 1. Introdução
O **NLPGym** transforma problemas de Processamento de Linguagem Natural (NLP) em ambientes de **Reinforcement Learning** seguindo a API do Gym/Gymnasium. Isso permite treinar agentes que **decidem** passo a passo enquanto processam texto.

### Por que importa?
- Permite *reward shaping* e políticas ativas em NLP.
- Facilita benchmarking padronizado de RL em texto.
- Oferece data pools prontos de benchmarks conhecidos.

### Tarefas suportadas
| Tarefa | Descrição |
|---|---|
| **Question Answering (QA)** | Pergunta + contexto + alternativas. O agente decide quando parar de ler opções e qual marcar. Recompensa = 1 se acertar. |
| **Sequence Tagging** | Etiquetagem token‑a‑token (NER, POS). Cada passo rotula um token. Recompensa densa ou F1 final. |
| **Multi‑Label Classification** | Texto pode ter vários rótulos. O agente adiciona rótulos um por vez até terminar. |


## 2. Passo a Passo de Uso
1. **Instalar** bibliotecas:
   ```bash
   pip install nlp-gym stable-baselines3 gymnasium torch
   ```
2. **Preparar** um *data pool* (ex.: `QASC`).
3. **Criar** o ambiente e alimentar com amostras.
4. **Envolver** com `EnvCompatibility`, `Monitor` e `DummyVecEnv`.
5. **Instanciar** agente (DQN, PPO, A2C…).
6. **Treinar** e **avaliar**.


### 2.1 Wrapper de Compatibilidade

In [ ]:

import gymnasium as gym
from gymnasium import spaces as gs
import gym as ogym

class CompatEnv(gym.Env):
    """Wrapper que adapta envs Gym antigos para Gymnasium."""
    def __init__(self, env):
        super().__init__()
        self.env = env
        self.action_space = self._convert(env.action_space)
        self.observation_space = self._convert(env.observation_space)
    def _convert(self, space):
        if isinstance(space, ogym.spaces.Box):
            return gs.Box(low=space.low, high=space.high, dtype=space.dtype)
        if isinstance(space, ogym.spaces.Discrete):
            return gs.Discrete(space.n)
        if isinstance(space, ogym.spaces.Tuple):
            return gs.Tuple(tuple(self._convert(s) for s in space.spaces))
        if isinstance(space, ogym.spaces.Dict):
            return gs.Dict({k:self._convert(v) for k,v in space.spaces.items()})
        return space
    def reset(self,* ,seed=None, **kw):
        obs = self.env.reset()
        return obs, {}
    def step(self, action):
        obs, reward, done, info = self.env.step(action)
        return obs, reward, done, False, info
    def render(self,*a,**k):
        return self.env.render(*a,**k)
    def close(self):
        self.env.close()


## 3. Exemplos Robustos para Cada Tarefa

### 3.1 Question Answering – DQN (CPU)

In [ ]:

from nlp_gym.data_pools.custom_question_answering_pools import QASC
from nlp_gym.envs.question_answering.env import QAEnv
from nlp_gym.envs.question_answering.featurizer import InformedFeaturizer
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3 import DQN
import os, torch

# preparar dados
train_pool = QASC.prepare('train')
env_raw = QAEnv(observation_featurizer=InformedFeaturizer())
for s,w in train_pool:
    env_raw.add_sample(s,w)

env = DummyVecEnv([lambda: Monitor(CompatEnv(env_raw))])

model = DQN('MlpPolicy', env, learning_rate=1e-4, batch_size=32, verbose=0, device='cpu')
model.learn(total_timesteps=20000)
os.makedirs('models', exist_ok=True)
model.save('models/dqn_qa_demo')
print('✅ DQN QA treinado (CPU)')


### 3.2 Sequence Tagging – PPO (CPU)

In [ ]:

from nlp_gym.data_pools.seq_tagging_pools import CoNLL2003
from nlp_gym.envs.seq_tagging.env import SequenceTaggingEnv
from stable_baselines3 import PPO

train_pool_st = CoNLL2003.prepare('train')
env_st_raw = SequenceTaggingEnv()
for s,_ in train_pool_st:
    env_st_raw.add_sample(s)

vec_env_st = DummyVecEnv([lambda: Monitor(CompatEnv(env_st_raw))])
model_st = PPO('MlpPolicy', vec_env_st, n_steps=1024, batch_size=128, verbose=0, device='cpu')
model_st.learn(total_timesteps=10000)
model_st.save('models/ppo_seqtag_demo')
print('✅ PPO Sequence Tagging treinado')


### 3.3 Multi‑Label Classification – A2C (CPU)

In [ ]:

from nlp_gym.data_pools.multi_label_pools import Reuters
from nlp_gym.envs.multi_label.env import MultiLabelEnv
from stable_baselines3 import A2C

train_pool_ml = Reuters.prepare('train')
env_ml_raw = MultiLabelEnv()
for s,_ in train_pool_ml:
    env_ml_raw.add_sample(s)

vec_env_ml = DummyVecEnv([lambda: Monitor(CompatEnv(env_ml_raw))])
model_ml = A2C('MlpPolicy', vec_env_ml, verbose=0, device='cpu')
model_ml.learn(total_timesteps=15000)
model_ml.save('models/a2c_multilabel_demo')
print('✅ A2C Multi‑Label treinado')


## 4. Exemplo Opcional com GPU

In [ ]:

import torch, os
device='cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', device)
if device=='cuda':
    # Reutiliza env QA
    model_gpu = DQN('MlpPolicy', env, learning_rate=1e-4, batch_size=32, device=device, verbose=0)
    model_gpu.learn(total_timesteps=50000)
    model_gpu.save('models/dqn_qa_gpu_demo')
    print('✅ Modelo QA treinado em GPU')
else:
    print('GPU não disponível – pule esta célula')


## 5. Próximos Passos
- Aumentar `total_timesteps`, ajustar hiperparâmetros.
- Experimentar reward shaping e embeddings mais ricos.
- Monitorar com TensorBoard (`tensorboard --logdir logs/`).